# 3. Combining directional cues

A migrating cell may respond to neighbors, soluble signals and an
oriented matrix at the same time. In BioLGCA, directional biases
that should compete in one decision belong as terms in one
reorientation sampler.

**Learning objectives**

- construct a `ReorientationSpec` from visible terms;
- combine alignment with chemotaxis;
- combine persistent motion with contact guidance;
- measure competition between cues with a parameter sweep; and
- distinguish additive terms from sequential pipeline phases.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
)
from lgca.pipeline import (
    InteractionPipelineSpec,
    ReorientationSpec,
    ReorientationTermSpec,
    list_reorientation_terms,
)
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder

print("available terms:", list_reorientation_terms())


## Alignment plus chemotaxis

We define a scalar signal that increases from left to right. The
alignment score depends on neighboring channel occupancy; the
chemotaxis score depends on the signal gradient. Both scores enter
the same Boltzmann distribution over admissible channel states.


In [ ]:
DIMS = (12, 12)
signal = np.linspace(0.0, 1.0, DIMS[0])[:, None] + np.zeros(DIMS)


def make_alignment_chemotaxis_spec(
    alignment_beta=1.0,
    chemotaxis_beta=1.0,
    seed=31,
    steps=10,
):
    return ModelSpec(
        description=Description(title="Alignment plus chemotaxis"),
        space=SpaceSpec(
            geometry="square",
            dims=DIMS,
            boundary="periodic",
        ),
        state=StateSpec(
            density=0.25,
            restchannels=0,
            fields={"signal": signal},
        ),
        time=TimeSpec(steps=steps, seed=seed),
        dynamics=InteractionPipelineSpec(
            operators=[
                ReorientationSpec(
                    terms=[
                        ReorientationTermSpec(
                            name="alignment",
                            beta=alignment_beta,
                        ),
                        ReorientationTermSpec(
                            name="chemotaxis",
                            beta=chemotaxis_beta,
                            parameters={"field": "signal"},
                        ),
                    ],
                )
            ],
        ),
        analysis=AnalysisSpec(
            observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
        ),
    )


combined = run_model(make_alignment_chemotaxis_spec(), showprogress=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), constrained_layout=True)
axes[0].imshow(signal.T, origin="lower", cmap="viridis")
axes[0].set_title("signal field")
axes[1].imshow(combined.lgca.dens_t[-1].T, origin="lower", cmap="magma")
axes[1].set_title("final cell density")
for axis in axes:
    axis.set_xlabel("x")
    axis.set_ylabel("y")
plt.show()
plt.close(fig)


The two terms do not update the lattice one after another. Their
weighted scores are added first, followed by **one sampled
reorientation transition** of the complete channel state.


## A controlled competition experiment

Mean flux in the x direction measures response to the signal. We
sweep the two beta values on a small grid. This is an exploratory
map, not a converged phase diagram; a paper would use more seeds and
a justified parameter range.


In [ ]:
def mean_x_flux(model_result):
    nodes = model_result.lgca.nodes_t[-1]
    total = nodes.sum()
    if total == 0:
        return 0.0
    flux = model_result.lgca.calc_flux(nodes)
    return float(flux[..., 0].sum() / total)


beta_values = (0.0, 1.0, 2.0)
response = np.zeros((len(beta_values), len(beta_values)))
for row, alignment_beta in enumerate(beta_values):
    for column, chemotaxis_beta in enumerate(beta_values):
        sweep_result = run_model(
            make_alignment_chemotaxis_spec(
                alignment_beta=alignment_beta,
                chemotaxis_beta=chemotaxis_beta,
                seed=32,
                steps=6,
            ),
            showprogress=False,
        )
        response[row, column] = mean_x_flux(sweep_result)

fig, axis = plt.subplots(figsize=(5, 4), constrained_layout=True)
image = axis.imshow(response, origin="lower", cmap="coolwarm")
axis.set_xticks(range(len(beta_values)), beta_values)
axis.set_yticks(range(len(beta_values)), beta_values)
axis.set_xlabel("chemotaxis beta")
axis.set_ylabel("alignment beta")
fig.colorbar(image, ax=axis, label="mean x flux")
plt.show()
plt.close(fig)


## Persistent motion plus contact guidance

A director field describes an oriented scaffold without a head or
tail. Persistent walk favors the current local direction, while
contact guidance favors the scaffold axis. Again, both terms appear
visibly in one sampler.


In [ ]:
director = np.zeros(DIMS + (2,), dtype=float)
director[..., 0] = 1.0

persistent_guidance_spec = ModelSpec(
    description=Description(title="Persistence plus contact guidance"),
    space=SpaceSpec(geometry="square", dims=DIMS, boundary="periodic"),
    state=StateSpec(
        density=0.2,
        restchannels=0,
        fields={"director": director},
    ),
    time=TimeSpec(steps=10, seed=35),
    dynamics=InteractionPipelineSpec(
        operators=[
            ReorientationSpec(
                terms=[
                    ReorientationTermSpec(
                        name="persistent_walk",
                        beta=1.0,
                    ),
                    ReorientationTermSpec(
                        name="contact_guidance",
                        beta=1.5,
                        parameters={"field": "director"},
                    ),
                ],
            )
        ],
    ),
    analysis=AnalysisSpec(
        observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
    ),
)

persistent_guidance = run_model(persistent_guidance_spec, showprogress=False)
print("mean final x flux:", mean_x_flux(persistent_guidance))


## Terms versus sequential pipeline operators

There are two distinct meanings of "combine interactions":

1. Multiple reorientation terms contribute to one energy score and
   one sampled channel-state transition. This is how directional
   biases reinforce or compete.
2. A sequential pipeline represents biologically different phases,
   such as birth/death, phenotype switching and then reorientation.

Placing two full reorientation operators sequentially performs two
stochastic transitions. The later transition does not merely add a
bias to the earlier one. Lesson 4 constructs a valid sequential
multi-phase pipeline.

## Exercises

1. Reverse the signal field and confirm that mean x flux changes sign.
2. Rotate the director field by 90 degrees and define a y-flux measure.
3. Repeat the beta sweep for several seeds and plot the mean and
   standard deviation at each parameter pair.
4. Combine `resting_bias` with chemotaxis after adding rest channels.
